# 02 - Build ECG-CXR pair manifests

This notebook reads only the CSV files bundled with Symile-MIMIC. Each JSONL row preserves the exact NPY row order and the same-patient/same-admission ECG-CXR pairing. CheXpert values are retained as metadata for stratified evaluation only; they are not training targets.

In [ ]:
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts",
))
config = json.loads((ARTIFACT_ROOT / "config.json").read_text())
SYMILE_ROOT = Path(config["symile_root"])
MANIFEST_ROOT = ARTIFACT_ROOT / "manifests"
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
CHEXPERT_COLUMNS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
                     "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion",
                     "Lung Opacity", "No Finding", "Pleural Effusion",
                     "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"]


In [ ]:
def study_id_from_path(value):
    matches = [part for part in Path(str(value)).parts if re.fullmatch(r"s\d+", part)]
    if len(matches) != 1:
        raise ValueError(f"Cannot identify one study ID in {value}")
    return int(matches[0][1:])

def observed_chexpert(row, columns):
    return {name: float(row[name]) for name in columns if name in row.index and pd.notna(row[name])}


In [ ]:
summary = {}
for split in config["splits"]:
    frame = pd.read_csv(SYMILE_ROOT / f"{split}.csv", low_memory=False)
    output_path = MANIFEST_ROOT / f"{split}.jsonl"
    available_labels = [name for name in CHEXPERT_COLUMNS if name in frame.columns]
    with output_path.open("w") as stream:
        for npy_index, row in tqdm(frame.iterrows(), total=len(frame), desc=split):
            record = {
                "npy_index": int(npy_index),
                "subject_id": int(row.subject_id),
                "hadm_id": int(row.hadm_id),
                "ecg_study_id": study_id_from_path(row.ecg_path),
                "cxr_study_id": study_id_from_path(row.cxr_path),
                "ecg_path": str(row.ecg_path),
                "cxr_path": str(row.cxr_path),
                "cxr_chexpert": observed_chexpert(row, available_labels),
            }
            if "label_hadm_id" in frame.columns:
                record["label_hadm_id"] = int(row.label_hadm_id)
                record["label"] = int(row.label)
            stream.write(json.dumps(record) + "\n")
    summary[split] = {"rows": len(frame), "available_chexpert_columns": available_labels}
    print(split, summary[split], "->", output_path)


In [ ]:
for split in config["splits"]:
    records = [json.loads(line) for line in (MANIFEST_ROOT / f"{split}.jsonl").read_text().splitlines()]
    ids = np.load(SYMILE_ROOT / "data_npy" / split / f"hadm_id_{split}.npy", mmap_mode="r")
    assert len(records) == len(ids)
    assert all(record["npy_index"] == index for index, record in enumerate(records))
    assert all(record["hadm_id"] == int(ids[index]) for index, record in enumerate(records))
print("All pair manifests match NPY row order.")
summary_path = MANIFEST_ROOT / "manifest_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("Saved:", summary_path)
